In [ ]:
import requests
import pandas as pd
import time
import os
import math

# Disable proxies if set
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)

# Configuration
API_KEY = 'f2cc020b5cb97a17399ff33316d74ac4'
hospital_file_path = r'/National hospital directory.xlsx'
output_base_path = r'/hospital location/hospital_location_batch{batch_num}.xlsx'


hospital_data = pd.read_excel(hospital_file_path, sheet_name="select").fillna('')
total_rows = len(hospital_data)
batch_size = 500
batch_count = math.ceil(total_rows / batch_size)

start_batch = 4  # Batch index to resume from

for batch_num in range(start_batch, batch_count):
    batch_start = batch_num * batch_size
    batch_end = min((batch_num + 1) * batch_size, total_rows)
    print(f"\n=== Processing batch {batch_num + 1}: rows {batch_start}–{batch_end - 1} ===")
    batch_data = hospital_data.iloc[batch_start:batch_end]

    results = []

    for idx, row in batch_data.iterrows():
        province = str(row['province']).strip()
        city     = str(row['city']).strip()
        hospital = str(row['hospital']).strip()
        address  = str(row['address']).strip()

        success     = False
        method_used = ''
        lng = lat   = None

        # Amap Geocoding API base URL
        url = 'https://restapi.amap.com/v3/geocode/geo'
        params = {
            'key': API_KEY,
            'city': city
        }

        # ---- Step 1: Query by hospital name ----
        params['address'] = f"{province}{hospital}"
        print(f"\n[Request] Method: hospital name | address: {params['address']} | city: {city}")
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            data   = resp.json()
            status = data.get('status')
            count  = int(data.get('count', 0))
            print(f"[Response] status={status}, count={count}")
            if status == '1' and count > 0:
                loc       = data['geocodes'][0]['location']
                lng, lat  = loc.split(',')
                print(f"[Match found] lng={lng}, lat={lat}")
                success     = True
                method_used = 'hospital name'
        except Exception as e:
            print(f"[Error] Request failed during hospital name lookup: {e}")

        # ---- Step 2: Fall back to detailed address if Step 1 failed ----
        if not success:
            params['address'] = f"{province}{address}"
            print(f"\n[Request] Method: address | address: {params['address']} | city: {city}")
            try:
                resp = requests.get(url, params=params, timeout=30)
                resp.raise_for_status()
                data   = resp.json()
                status = data.get('status')
                count  = int(data.get('count', 0))
                print(f"[Response] status={status}, count={count}")
                if status == '1' and count > 0:
                    loc       = data['geocodes'][0]['location']
                    lng, lat  = loc.split(',')
                    print(f"[Match found] lng={lng}, lat={lat}")
                    success     = True
                    method_used = 'address'
                else:
                    print("[No result] Address lookup returned no matches.")
            except Exception as e:
                print(f"[Error] Request failed during address lookup: {e}")

        results.append({
            'Province':      province,
            'City':          city,
            'Hospital name': hospital,
            'Address':       address,
            'Longitude':     float(lng) if success else None,
            'Latitude':      float(lat) if success else None,
            'Match method':  method_used if success else 'failed'
        })

        time.sleep(0.5)

    df_batch          = pd.DataFrame(results)
    batch_output_path = output_base_path.format(batch_num=batch_num + 1)
    df_batch.to_excel(batch_output_path, index=False)
    print(f"\n✅ Batch {batch_num + 1} saved: {batch_output_path}")


=== 处理第 5 批：行 2000–2499 ===

[请求] 方法：医院名，地址参数：辽宁省辽中县中医院，城市：辽中县
[返回] status=1, count=1
[匹配成功] 经度=122.766159, 纬度=41.517449

[请求] 方法：医院名，地址参数：辽宁省辽中县传染病院，城市：辽中县
[返回] status=1, count=1
[匹配成功] 经度=122.723533, 纬度=41.488624

[请求] 方法：医院名，地址参数：辽宁省康平县第二人民医院，城市：康平县
[返回] status=1, count=10
[匹配成功] 经度=123.343701, 纬度=42.727249

[请求] 方法：医院名，地址参数：辽宁省康平县中医院，城市：康平县
[返回] status=1, count=1
[匹配成功] 经度=123.350147, 纬度=42.749299

[请求] 方法：医院名，地址参数：辽宁省法库县第一医院，城市：法库县
[返回] status=1, count=1
[匹配成功] 经度=123.440294, 纬度=42.501080

[请求] 方法：医院名，地址参数：辽宁省法库县第二医院，城市：法库县
[返回] status=1, count=10
[匹配成功] 经度=123.440294, 纬度=42.501080

[请求] 方法：医院名，地址参数：辽宁省法库县中医院，城市：法库县
[返回] status=1, count=1
[匹配成功] 经度=123.427713, 纬度=42.504223

[请求] 方法：医院名，地址参数：辽宁省铁岭市妇婴医院，城市：铁岭市
[返回] status=1, count=1
[匹配成功] 经度=123.843178, 纬度=42.291077

[请求] 方法：医院名，地址参数：辽宁省铁岭市银州区医院，城市：铁岭市
[返回] status=1, count=1
[匹配成功] 经度=123.856242, 纬度=42.301006

[请求] 方法：医院名，地址参数：辽宁省铁岭市银州区中西医结合医院，城市：铁岭市
[返回] status=1, count=1
[匹配成功] 经度=123.852968, 纬度=42.299035

[请求] 方法：医院名，地址参数：辽宁省铁岭